# MORPHEUS: RNA-seq preprocessing for Gene Co-expression Graph (GCG)

We perform transcriptomic data preprocessing for the MORPHEUS framework, specifically tailored for graph-based analysis. The goal is to transform raw RNA-seq gene expression data into a structured format suitable for constructing a Gene Co-expression Graph (GCG)

## Objective
To generate a filtered and normalized RNA-seq matrix that preserves gene-level relationships for downstream graph construction and graph neural network (GNN) modeling

## Pipeline Overview

The preprocessing pipeline consists of the following steps:

1. **Cohort Assembly and Patient-Level Splitting**
   - Load patient IDs and LUAD/LUSC labels
   - Perform stratified split into train/validation/test (7:1:2)

2. **Raw RNA-seq Data Loading**
   - Load FPKM-UQ expression values
   - Align patient IDs across datasets

3. **Gene Filtering**
   - Remove genes with >80% zero expression
   - Retain top 25% most variable genes using Median Absolute Deviation (MAD)

4. **Data Transformation and Normalization**
   - Apply log₂(FPKM-UQ + 1) transformation
   - Perform per-sample z-score normalization

5. **Output Generation**
   - Save filtered RNA matrix (N × G)
   - Save gene identifiers
   - Save split-specific datasets (train/val/test)
   - Save preprocessing statistics for reproducibility


In [1]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

WORKDIR = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/GraphBased-Analysis")
os.chdir(WORKDIR)
print("CWD set to:", Path.cwd())

RNA_ROOTS = {
    "LUAD": Path(
        "/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/01_download/"
        "rna/TCGA-LUAD/Transcriptome_Profiling/Gene_Expression_Quantification"
    ),
    "LUSC": Path(
        "/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/01_download/"
        "rna/TCGA-LUSC/Transcriptome_Profiling/Gene_Expression_Quantification"
    ),
}

LABELS_PATH = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/multiomics_labels.tsv")

PROJECT_ROOT = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping")

OUT_BASE   = PROJECT_ROOT / "Data" / "02_preprocessed" / "rna_gcg"
SPLITS_DIR = PROJECT_ROOT / "Data" / "splits"

OUT_BASE.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

print("\n--- Paths ---")
print("RNA_ROOTS['LUAD']:", RNA_ROOTS["LUAD"])
print("RNA_ROOTS['LUSC']:", RNA_ROOTS["LUSC"])
print("LABELS_PATH:", LABELS_PATH)
print("OUT_BASE:", OUT_BASE)
print("SPLITS_DIR:", SPLITS_DIR)

if not WORKDIR.exists():
    raise FileNotFoundError(f"WORKDIR not found: {WORKDIR}")

for k, p in RNA_ROOTS.items():
    if not p.exists():
        raise FileNotFoundError(f"Missing RNA root for {k}: {p}")

if not LABELS_PATH.exists():
    raise FileNotFoundError(f"Missing labels file: {LABELS_PATH}")

print("\nAll required paths exist.")

CWD set to: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/GraphBased-Analysis

--- Paths ---
RNA_ROOTS['LUAD']: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/01_download/rna/TCGA-LUAD/Transcriptome_Profiling/Gene_Expression_Quantification
RNA_ROOTS['LUSC']: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/01_download/rna/TCGA-LUSC/Transcriptome_Profiling/Gene_Expression_Quantification
LABELS_PATH: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/multiomics_labels.tsv
OUT_BASE: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/rna_gcg
SPLITS_DIR: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/splits

All required paths exist.


### Load Labels, Build Cohort, and Create 7:1:2 Stratified Split

We:

- Load the cohort label file (`multiomics_labels.tsv`) and identify the columns that contain patient identifiers and cancer subtype labels.
- Restrict the cohort to **NSCLC subtypes**: **LUAD** and **LUSC**.
- Encode labels as integers (**LUAD = 0**, **LUSC = 1**).
- Perform a **patient-level stratified split** into **train/validation/test = 7:1:2**, ensuring class balance across splits.
- Save split patient IDs and labels to disk so that the *same splits* can be reused consistently across all modality-specific preprocessing notebooks.

In [2]:
from sklearn.model_selection import train_test_split

# Load labels tabel
labels = pd.read_csv(LABELS_PATH, sep="\t")
print("Labels shape:", labels.shape)
print(labels.head())

Labels shape: (831, 3)
     patient_id    subtype subtype_simple
0  TCGA-MP-A4SV  TCGA-LUAD           LUAD
1  TCGA-55-8621  TCGA-LUAD           LUAD
2  TCGA-MN-A4N1  TCGA-LUAD           LUAD
3  TCGA-55-6986  TCGA-LUAD           LUAD
4  TCGA-86-6851  TCGA-LUAD           LUAD


In [3]:
# Stratified Split (7:1:2) 

labels = labels[labels["subtype_simple"].isin(["LUAD", "LUSC"])].copy()

# Encode labels
label_map = {"LUAD": 0, "LUSC": 1}
labels["y"] = labels["subtype_simple"].map(label_map).astype(int)

pids = labels["patient_id"].astype(str).values
y    = labels["y"].values

# 7:1:2 split (patient-level, stratified)
train_pids, temp_pids, y_train, y_temp = train_test_split(
    pids, y, test_size=0.30, random_state=SEED, stratify=y
)

val_pids, test_pids, y_val, y_test = train_test_split(
    temp_pids, y_temp, test_size=2/3, random_state=SEED, stratify=y_temp
)

print("Split sizes:", {"train": len(train_pids), "val": len(val_pids), "test": len(test_pids)})
print("Class balance:")
print("  train:", dict(zip(*np.unique(y_train, return_counts=True))))
print("  val  :", dict(zip(*np.unique(y_val,   return_counts=True))))
print("  test :", dict(zip(*np.unique(y_test,  return_counts=True))))

# Save for reuse across modalities
np.save(SPLITS_DIR / "train_pids.npy", train_pids)
np.save(SPLITS_DIR / "val_pids.npy",   val_pids)
np.save(SPLITS_DIR / "test_pids.npy",  test_pids)

np.save(SPLITS_DIR / "y_train.npy", y_train)
np.save(SPLITS_DIR / "y_val.npy",   y_val)
np.save(SPLITS_DIR / "y_test.npy",  y_test)

print("Saved splits to:", SPLITS_DIR)

Split sizes: {'train': 581, 'val': 83, 'test': 167}
Class balance:
  train: {np.int64(0): np.int64(321), np.int64(1): np.int64(260)}
  val  : {np.int64(0): np.int64(46), np.int64(1): np.int64(37)}
  test : {np.int64(0): np.int64(92), np.int64(1): np.int64(75)}
Saved splits to: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/splits


### Load RNA Expression Data and Align with Cohort Splits

We:

- Load raw RNA-seq expression files (FPKM-UQ) for LUAD and LUSC cohorts.
- Extract gene expression vectors for each patient.
- Match expression samples with patient IDs from the previously created splits.
- Construct a unified expression matrix **X (N × G)** where:
  - **N** = number of patients
  - **G** = number of genes (before filtering)
- Ensure consistent ordering between features and labels.

This matrix will serve as the input for downstream preprocessing and gene co-expression graph construction.

##### Collect RNA Count Files (UUID folders → TSV paths)

We:

- Enumerate the UUID-named folders under each cohort RNA directory (LUAD/LUSC).
- Locate the corresponding RNA quantification file inside each folder:
  `*.rna_seq.augmented_star_gene_counts.tsv`
- Build a table of RNA files with their cohort and UUID folder identifiers.

This step prepares the file list needed for mapping UUIDs to TCGA patient IDs in the next cell.

In [6]:
def collect_rna_files(root: Path):
    uuid_folders = [p for p in root.iterdir() if p.is_dir()]
    files = []
    for folder in uuid_folders:
        hits = list(folder.glob("*.rna_seq.augmented_star_gene_counts.tsv"))
        if len(hits) == 1:
            files.append({"uuid_folder": folder.name, "path": hits[0]})
    return uuid_folders, files

rna_uuid_folders = {}
rna_files = []

for cancer, root in RNA_ROOTS.items():
    folders, files = collect_rna_files(root)
    rna_uuid_folders[cancer] = folders
    for d in files:
        d["cancer"] = cancer
        rna_files.append(d)

print("LUAD uuid folders:", len(rna_uuid_folders["LUAD"]))
print("LUSC uuid folders:", len(rna_uuid_folders["LUSC"]))
print("Total RNA count TSV files found:", len(rna_files))
print("\nFirst 3 examples:")
for x in rna_files[:3]:
    print(x["cancer"], x["uuid_folder"], "->", x["path"].name)

LUAD uuid folders: 455
LUSC uuid folders: 370
Total RNA count TSV files found: 825

First 3 examples:
LUAD ffec28dd-cc44-4057-bcc3-ccc0cd2331c2 -> d21d91e9-6e19-4c5f-aa2e-cdd41d98e542.rna_seq.augmented_star_gene_counts.tsv
LUAD fe20855f-d3ce-4e02-b2dd-3df15eab44bf -> 564e41d6-f9a4-4a6b-bb97-f8e5423b84bd.rna_seq.augmented_star_gene_counts.tsv
LUAD fdcad604-c1b4-40ff-a652-3c1e5fe5f9a1 -> 2301103f-5cbe-4cb6-b5da-0b83b39e4616.rna_seq.augmented_star_gene_counts.tsv


Map RNA UUID to patient_id using the GDC API

In [11]:
RNA_META_PATH = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/RNASeq_metadata.tsv")
meta = pd.read_csv(RNA_META_PATH, sep="\t", dtype=str)

print("Metadata shape:", meta.shape)
print("Columns:", meta.columns.tolist())

Metadata shape: (1, 100)
Columns: ['barcode', 'patient', 'sample', 'shortLetterCode', 'definition', 'sample_submitter_id', 'tumor_descriptor', 'specimen_type', 'sample_id', 'submitter_id', 'state', 'sample_type', 'tissue_type', 'preservation_method', 'intermediate_dimension', 'pathology_report_uuid', 'shortest_dimension', 'longest_dimension', 'days_to_collection', 'initial_weight', 'synchronous_malignancy', 'ajcc_pathologic_stage', 'days_to_diagnosis', 'laterality', 'treatments', 'tissue_or_organ_of_origin', 'age_at_diagnosis', 'primary_diagnosis', 'prior_malignancy', 'year_of_diagnosis', 'prior_treatment', 'diagnosis_is_primary_disease', 'ajcc_staging_system_edition', 'ajcc_pathologic_t', 'morphology', 'ajcc_pathologic_n', 'ajcc_pathologic_m', 'residual_disease', 'classification_of_tumor', 'diagnosis_id', 'icd_10_code', 'site_of_resection_or_biopsy', 'sites_of_involvement', 'tumor_of_origin', 'tobacco_smoking_status', 'exposure_id', 'exposure_type', 'tobacco_smoking_onset_year', 'pack

In [12]:
def pick_col(cols, candidates):
    cols_lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    # fallback: partial match
    for c in cols:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

uuid_col = pick_col(meta.columns, ["file_id", "id", "file_uuid", "uuid"])
patient_col = pick_col(meta.columns, ["cases.0.submitter_id", "cases.submitter_id", "submitter_id", "case_submitter_id", "patient_id"])

print("Detected uuid_col:", uuid_col)
print("Detected patient_col:", patient_col)

if uuid_col is None or patient_col is None:
    raise ValueError("Could not auto-detect uuid/patient columns. Paste meta.columns.tolist() output and we’ll map manually.")

Detected uuid_col: sample_submitter_id
Detected patient_col: submitter_id


In [13]:
import re

uuid_re = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$", re.I)

def looks_like_uuid_series(s):
    s = s.dropna().astype(str).str.strip()
    if len(s) == 0:
        return 0
    return (s.apply(lambda x: bool(uuid_re.match(x)))).mean()

scores = []
for c in meta.columns:
    frac = looks_like_uuid_series(meta[c])
    if frac > 0:
        scores.append((c, frac))

scores = sorted(scores, key=lambda x: x[1], reverse=True)
print("UUID-like columns (top):")
for c, frac in scores[:10]:
    print(f"{c:40s}  uuid_fraction={frac:.3f}")

UUID-like columns (top):
submitter_id                              uuid_fraction=1.000
shortest_dimension                        uuid_fraction=1.000


In [14]:
# quick peek at what those "uuid-like" columns actually contain
for c in ["submitter_id", "shortest_dimension"]:
    if c in meta.columns:
        vals = meta[c].dropna().astype(str).str.strip().head(10).tolist()
        print(f"\nColumn: {c}")
        for v in vals:
            print("  ", v)


Column: submitter_id
   b5c9beed-da9d-4110-9b6d-54a66498d5cc

Column: shortest_dimension
   8E809652-9783-4B7A-B92D-8C393A2940AB


In [15]:
def tcga_case_fraction(s):
    s = s.dropna().astype(str).str.strip()
    if len(s) == 0:
        return 0
    return s.str.match(r"^TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}$").mean()

tcga_scores = []
for c in meta.columns:
    frac = tcga_case_fraction(meta[c])
    if frac > 0:
        tcga_scores.append((c, frac))

tcga_scores = sorted(tcga_scores, key=lambda x: x[1], reverse=True)

print("TCGA-case-like columns (top):")
for c, frac in tcga_scores[:10]:
    print(f"{c:40s}  tcga_case_fraction={frac:.3f}")

TCGA-case-like columns (top):
sample                                    tcga_case_fraction=1.000
state                                     tcga_case_fraction=1.000


In [16]:
# Define correct columns
uuid_col = "submitter_id"   # UUID (file-level)
patient_col = "sample"      # TCGA patient ID

# Build mapping
m = meta[[uuid_col, patient_col]].dropna().copy()
m[uuid_col] = m[uuid_col].astype(str).str.strip()
m[patient_col] = m[patient_col].astype(str).str.strip()

uuid_to_patient = dict(zip(m[uuid_col], m[patient_col]))

print("UUID → patient mappings:", len(uuid_to_patient))
print("Example mappings:")
for k, v in list(uuid_to_patient.items())[:5]:
    print(k, "→", v)

UUID → patient mappings: 1
Example mappings:
b5c9beed-da9d-4110-9b6d-54a66498d5cc → TCGA-MP-A4SV


In [19]:
# Attach patient IDs to RNA files
annotated = []

for d in rna_files:
    uuid = d["uuid_folder"]
    patient = uuid_to_patient.get(uuid)
    if patient is not None:
        annotated.append({
            "cancer": d["cancer"],
            "uuid": uuid,
            "patient_id": patient,
            "path": d["path"]
        })

rna_df = pd.DataFrame(annotated)

print("RNA files with patient IDs:", rna_df.shape[0])
rna_df.head()

RNA files with patient IDs: 0


""


In [20]:
# show UUID-like columns with examples
for c in meta.columns:
    if meta[c].astype(str).str.match(
        r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}",
        na=False
    ).mean() > 0.5:
        print(c)
        print(meta[c].dropna().iloc[:3])
        print()

submitter_id
0    b5c9beed-da9d-4110-9b6d-54a66498d5cc
Name: submitter_id, dtype: object



Locate the Correct UUID → Patient Mapping File

The current `RNASeq_metadata.tsv` is not the RNA download sample sheet (it contains only one record),
so it cannot map the 825 UUID folders to TCGA patient IDs.

In this cell, we scan the `Data/` directory for a manifest/sample-sheet/metadata file that contains
the RNA UUIDs (folder names) and a patient identifier. We then select the best candidate for mapping.

In [22]:
DATA_DIR = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data")

# sample a few UUID folder names for fast searching
uuid_sample = [d["uuid_folder"] for d in rna_files[:30]]
print("UUID sample (first 5):", uuid_sample[:5])

# candidate files to scan (prioritize likely mapping sources)
candidates = []
for ext in ["*.tsv", "*.csv", "*.json"]:
    candidates.extend(DATA_DIR.rglob(ext))

priority = []
for p in candidates:
    name = p.name.lower()
    if any(k in name for k in ["manifest", "metadata", "sample", "sheet", "cart", "gdc"]):
        priority.append(p)

print("Candidate files (priority):", len(priority))

def contains_any_uuid_text(path: Path, uuids):
    try:
        txt = path.read_text(errors="ignore")
        return any(u in txt for u in uuids)
    except:
        return False

hits = [p for p in priority if contains_any_uuid_text(p, uuid_sample)]
print("\nFiles containing UUID sample:", len(hits))
for p in hits[:40]:
    print(" -", p)

# If nothing found in priority list, broaden the scan
if len(hits) == 0:
    print("\nNo hits in priority files — broad scanning all TSV/CSV/JSON (may take a bit)...")
    hits = [p for p in candidates if contains_any_uuid_text(p, uuid_sample)]
    print("Files containing UUID sample:", len(hits))
    for p in hits[:40]:
        print(" -", p)

UUID sample (first 5): ['ffec28dd-cc44-4057-bcc3-ccc0cd2331c2', 'fe20855f-d3ce-4e02-b2dd-3df15eab44bf', 'fdcad604-c1b4-40ff-a652-3c1e5fe5f9a1', 'fd52d8a9-e044-4f17-861a-9780752b20ee', 'fc757476-1d31-470b-b648-79608b201bdc']
Candidate files (priority): 1

Files containing UUID sample: 0

No hits in priority files — broad scanning all TSV/CSV/JSON (may take a bit)...
Files containing UUID sample: 1
 - /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/id_maps/file_uuid_to_patient_id.csv


Map RNA UUID Folders to TCGA Patient IDs

We:

- Load the existing UUID-to-patient mapping file:
  `id_maps/file_uuid_to_patient_id.csv`
- Build a dictionary mapping each RNA UUID folder → TCGA patient ID
- Attach patient IDs to the collected RNA file list (825 files)
- Retain only successfully mapped files and verify coverage

This step enables construction of the patient-aligned RNA expression matrix in the next cell.

In [23]:
MAP_PATH = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/id_maps/file_uuid_to_patient_id.csv")
if not MAP_PATH.exists():
    raise FileNotFoundError(f"Mapping file not found: {MAP_PATH}")

uuid_map = pd.read_csv(MAP_PATH, dtype=str)
print("uuid_map shape:", uuid_map.shape)
print("uuid_map columns:", uuid_map.columns.tolist())

# Try common column name patterns
uuid_candidates = ["file_uuid", "file_id", "uuid", "id", "file"]
pid_candidates  = ["patient_id", "case_submitter_id", "submitter_id", "patient", "bcr_patient_barcode"]

uuid_col = next((c for c in uuid_map.columns if c.lower() in uuid_candidates), None)
pid_col  = next((c for c in uuid_map.columns if c.lower() in pid_candidates), None)

# If the above fails (because columns are named differently), do a more flexible match:
if uuid_col is None:
    uuid_col = next((c for c in uuid_map.columns if "uuid" in c.lower() or "file" in c.lower()), None)
if pid_col is None:
    pid_col = next((c for c in uuid_map.columns if "patient" in c.lower() or "submitter" in c.lower() or "case" in c.lower()), None)

if uuid_col is None or pid_col is None:
    raise ValueError(f"Could not detect uuid/patient columns in {MAP_PATH}. Columns: {uuid_map.columns.tolist()}")

print("Using uuid column:", uuid_col)
print("Using pid  column:", pid_col)

# Build mapping dict: UUID folder → TCGA patient ID (first 12 chars)
uuid_to_patient = dict(
    zip(uuid_map[uuid_col].astype(str).str.strip(),
        uuid_map[pid_col].astype(str).str.strip().str[:12])
)

# Attach patient IDs to rna_files list from Cell 3
rna_files_df = pd.DataFrame(rna_files).copy()
rna_files_df["uuid_folder"] = rna_files_df["uuid_folder"].astype(str).str.strip()
rna_files_df["patient_id"] = rna_files_df["uuid_folder"].map(uuid_to_patient)

mapped = rna_files_df["patient_id"].notna().sum()
print(f"Mapped files: {mapped}/{len(rna_files_df)}")

# Drop unmapped
rna_files_df = rna_files_df.dropna(subset=["patient_id"]).copy()

# Keep only patients in our splits (train/val/test)
split_pids = set(np.concatenate([train_pids, val_pids, test_pids]).astype(str))
rna_files_df = rna_files_df[rna_files_df["patient_id"].isin(split_pids)].copy()

print("After restricting to split patients:", rna_files_df.shape)
print(rna_files_df.head())

# Save for reproducibility
out_map_used = OUT_BASE / "rna_files_mapped.tsv"
rna_files_df.to_csv(out_map_used, sep="\t", index=False)
print("Saved mapped RNA file table:", out_map_used)

uuid_map shape: (1668, 3)
uuid_map columns: ['file_id', 'case_submitter_id', 'patient_id']
Using uuid column: file_id
Using pid  column: case_submitter_id
Mapped files: 825/825
After restricting to split patients: (825, 4)
                            uuid_folder  \
0  ffec28dd-cc44-4057-bcc3-ccc0cd2331c2   
1  fe20855f-d3ce-4e02-b2dd-3df15eab44bf   
2  fdcad604-c1b4-40ff-a652-3c1e5fe5f9a1   
3  fd52d8a9-e044-4f17-861a-9780752b20ee   
4  fc757476-1d31-470b-b648-79608b201bdc   

                                                path cancer    patient_id  
0  /home/steps4growth/gmriechi/Lung-Cancer-Subtyp...   LUAD  TCGA-91-6829  
1  /home/steps4growth/gmriechi/Lung-Cancer-Subtyp...   LUAD  TCGA-69-8453  
2  /home/steps4growth/gmriechi/Lung-Cancer-Subtyp...   LUAD  TCGA-50-5055  
3  /home/steps4growth/gmriechi/Lung-Cancer-Subtyp...   LUAD  TCGA-62-A46V  
4  /home/steps4growth/gmriechi/Lung-Cancer-Subtyp...   LUAD  TCGA-MP-A4TJ  
Saved mapped RNA file table: /home/steps4growth/gmriechi/Lung-

#### Load FPKM-UQ Expression Vectors and Build the Raw RNA Matrix

We:

- Read each `*.rna_seq.augmented_star_gene_counts.tsv` file.
- Extract the gene-level expression column **FPKM-UQ (unstranded)** (preferred: `fpkm_uq_unstranded`).
- Remove STAR summary rows beginning with `__` (non-gene entries).
- Construct the raw RNA expression matrix **X_raw (N × G_raw)** aligned to TCGA patient IDs.

Outputs from this cell:
- `X_raw` (raw expression; before filtering)
- `gene_ids` (Ensembl gene identifiers; length = G_raw)
- `patient_ids` (order aligned to X_raw)

In [24]:
def read_rna_fpkm_uq(tsv_path: Path):
    """
    Reads an augmented STAR gene counts TSV and returns:
      gene_ids (np.ndarray[str]), expr (np.ndarray[float32])
    Uses fpkm_uq_unstranded when available.
    Drops STAR summary rows starting with '__'.
    """
    df = pd.read_csv(tsv_path, sep="\t", low_memory=False)

    # gene id column is usually first
    gene_col = df.columns[0]

    # prefer fpkm_uq_unstranded
    if "fpkm_uq_unstranded" in df.columns:
        val_col = "fpkm_uq_unstranded"
    elif "FPKM-UQ" in df.columns:
        val_col = "FPKM-UQ"
    elif "fpkm_uq" in df.columns:
        val_col = "fpkm_uq"
    else:
        raise ValueError(f"No FPKM-UQ column found in {tsv_path.name}. Columns: {df.columns.tolist()}")

    # remove summary rows
    df[gene_col] = df[gene_col].astype(str)
    df = df[~df[gene_col].str.startswith("__")].copy()

    # coerce numeric
    df[val_col] = pd.to_numeric(df[val_col], errors="coerce")
    df = df.dropna(subset=[val_col])

    gene_ids = df[gene_col].astype(str).values
    expr = df[val_col].astype(np.float32).values
    return gene_ids, expr


# Build a patient_id -> file_path mapping
pid_to_path = dict(zip(rna_files_df["patient_id"].astype(str), rna_files_df["path"].map(Path)))

# We will build X in the order: train, val, test (so everything aligns later)
ordered_pids = np.concatenate([train_pids, val_pids, test_pids]).astype(str)

X_list = []
genes_ref = None
kept_pids = []

n_missing = 0
for pid in ordered_pids:
    if pid not in pid_to_path:
        n_missing += 1
        continue

    g, x = read_rna_fpkm_uq(pid_to_path[pid])

    if genes_ref is None:
        genes_ref = g
    else:
        # ensure consistent gene length/order
        if len(g) != len(genes_ref):
            raise ValueError(f"Gene length mismatch for {pid}: {len(g)} vs {len(genes_ref)}")
        # If you want strict order matching, uncomment:
        # if not np.all(g == genes_ref):
        #     raise ValueError(f"Gene order mismatch for {pid}")

    X_list.append(x)
    kept_pids.append(pid)

print("Missing patients with no RNA file:", n_missing)
X_raw = np.vstack(X_list)
patient_ids = np.array(kept_pids)
gene_ids = np.array(genes_ref)

print("X_raw shape:", X_raw.shape)
print("patient_ids:", patient_ids.shape, "first 5:", patient_ids[:5])
print("gene_ids   :", gene_ids.shape, "first 5:", gene_ids[:5])

# Save raw artifacts 
np.save(OUT_BASE / "rna_raw_X.npy", X_raw)
np.save(OUT_BASE / "rna_raw_patient_ids.npy", patient_ids)
np.save(OUT_BASE / "rna_raw_gene_ids.npy", gene_ids)

print("Saved raw RNA artifacts to:", OUT_BASE)

ValueError: No FPKM-UQ column found in 593623b6-0f7d-4089-8697-518c573c86c6.rna_seq.augmented_star_gene_counts.tsv. Columns: ['# gene-model: GENCODE v36']